In [3]:
# -*- coding: utf-8 -*-
"""s3_nli_consistency.py

S3 — Output-vs-output NLI consistency (random-draw axis), document-level.
Reads the demo_sensitivity_runner.py JSONLs and computes, per test instance,
bidirectional NLI over the C(5,2)=10 unordered pairs of the K=5 seed outputs.
NLI is directional, so each pair is scored twice (A->B and B->A) = 20 forward
passes per instance.

Per pair (A,B), with e = P(entailment), c = P(contradiction):
    composite  s(A,B) = 0.5*(e_fwd + e_bwd) - max(c_fwd, c_bwd)
    pair is "contradictory" if max(c_fwd, c_bwd) > CONTRA_THRESHOLD

Reported per instance (then aggregated per task, per Experiment_Plan.md S3):
    mean composite s, mean entailment, mean contradiction,
    frac-contradiction-pairs.

Rules (matching s2_bert_consistency.py):
  - everything per instance first, aggregate after
  - empty/whitespace outputs excluded from pairing (UNKNOWN handling = S4);
    <2 valid outputs -> NaN
  - texts truncated by the tokenizer at MAX_LENGTH (document-level pass;
    claim-level DocLens-style scoring for long-note tasks is a separate script)

Outputs (in OUT_DIR):
  s3_nli_{MODEL_SLUG}_per_pair.csv      — one row per task x instance x pair
  s3_nli_{MODEL_SLUG}_per_instance.csv  — one row per task x instance
  s3_nli_{MODEL_SLUG}_summary.csv       — one row per task

Run:
    pip install transformers torch sentencepiece pandas
    python s3_nli_consistency.py
"""

import os
import json
import itertools
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------------------
# Config
# ------------------------------------------------------------------------

MODEL_NAME = "epfl-llm/meditron-7b"   # <- must match the runner's MODEL_NAME
MODEL_SLUG = MODEL_NAME.split('/')[-1].lower().replace('-', '_').replace('.', '_')

OUT_DIR = '/workspace/demo_sensitivity_runs'

# DeBERTa-v3-large NLI backbone (Phase 0: DeBERTa-v3-large-MNLI primary;
# RoBERTa-large-MNLI ablation = V3, swap NLI_MODEL and rerun).
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
MAX_LENGTH = 512
BATCH_SIZE = 16
CONTRA_THRESHOLD = 0.5   # pair flagged contradictory if max(c_fwd, c_bwd) > this

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

with open(os.path.join(OUT_DIR, 'config_frozen.json')) as f:
    CFG = json.load(f)
TASKS, SEEDS = CFG['tasks'], CFG['seeds']

# ------------------------------------------------------------------------
# Load runner outputs (same logic as S2)
# ------------------------------------------------------------------------

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]


def load_condition_outputs(task):
    """{seed: {instance_id: output_text}} for the 5 random-seed conditions."""
    per_seed = {}
    for s in SEEDS:
        path = os.path.join(OUT_DIR, f"gen_{MODEL_SLUG}_{task}_random_seed{s}.jsonl")
        if not os.path.exists(path):
            print(f"  [skip] missing: {os.path.basename(path)}")
            return None
        per_seed[s] = {r['instance_id']: r['output_text'] for r in load_jsonl(path)}
    return per_seed

# ------------------------------------------------------------------------
# NLI scorer
# ------------------------------------------------------------------------

from transformers import AutoTokenizer, AutoModelForSequenceClassification

print(f"Loading NLI model {NLI_MODEL} on {DEVICE} ...")
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()

# Map label indices robustly from the model config (order varies by checkpoint)
id2label = {i: l.lower() for i, l in nli.config.id2label.items()}
ENT_IDX = next(i for i, l in id2label.items() if 'entail' in l)
CON_IDX = next(i for i, l in id2label.items() if 'contra' in l)
print(f"  labels: {id2label}  (entail={ENT_IDX}, contra={CON_IDX})")


@torch.no_grad()
def nli_probs(premises, hypotheses):
    """Batched (P(entail), P(contra)) for directed premise->hypothesis pairs."""
    ents, cons = [], []
    for i in range(0, len(premises), BATCH_SIZE):
        enc = tokenizer(premises[i:i + BATCH_SIZE], hypotheses[i:i + BATCH_SIZE],
                        truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors='pt').to(DEVICE)
        probs = torch.softmax(nli(**enc).logits, dim=-1).cpu().numpy()
        ents.append(probs[:, ENT_IDX])
        cons.append(probs[:, CON_IDX])
    return np.concatenate(ents), np.concatenate(cons)

# ------------------------------------------------------------------------
# Score
# ------------------------------------------------------------------------

per_pair_rows, per_instance_rows = [], []

for task in TASKS:
    print(f"\n=== {task} ===")
    per_seed = load_condition_outputs(task)
    if per_seed is None:
        continue

    instance_ids = sorted(set.intersection(*(set(d) for d in per_seed.values())))
    print(f"  {len(instance_ids)} instances with all {len(SEEDS)} seed outputs")

    # Collect directed pairs for the whole task -> batched NLI calls.
    # Unordered pair (A,B) occupies directed slots (A->B) and (B->A).
    prem, hyp, owner = [], [], []   # owner[j] = (instance_id, pair_idx)
    inst_info = {}
    for iid in instance_ids:
        outs = [per_seed[s][iid] for s in SEEDS]
        valid = [o for o in outs if o and o.strip()]
        inst_info[iid] = {'n_valid': len(valid), 'n_empty': len(outs) - len(valid)}
        for p_idx, (a, b) in enumerate(itertools.combinations(valid, 2)):
            prem.append(a); hyp.append(b); owner.append((iid, p_idx, 'fwd'))
            prem.append(b); hyp.append(a); owner.append((iid, p_idx, 'bwd'))

    print(f"  scoring {len(prem)} directed pairs ...")
    if prem:
        E, C = nli_probs(prem, hyp)
    else:
        E, C = np.array([]), np.array([])

    # Reassemble directed scores into unordered pairs
    directed = {}
    for (iid, p_idx, d), e, c in zip(owner, E, C):
        directed[(iid, p_idx, d)] = (float(e), float(c))

    inst_pairs = {}
    for iid in instance_ids:
        n_valid = inst_info[iid]['n_valid']
        n_pairs = n_valid * (n_valid - 1) // 2
        pairs = []
        for p_idx in range(n_pairs):
            e_f, c_f = directed[(iid, p_idx, 'fwd')]
            e_b, c_b = directed[(iid, p_idx, 'bwd')]
            s = 0.5 * (e_f + e_b) - max(c_f, c_b)
            contra = max(c_f, c_b) > CONTRA_THRESHOLD
            pairs.append({'s': s, 'e': 0.5 * (e_f + e_b),
                          'c': max(c_f, c_b), 'contra': contra})
            per_pair_rows.append({
                'model': MODEL_NAME, 'task': task, 'instance_id': iid,
                'pair_idx': p_idx,
                'entail_fwd': e_f, 'entail_bwd': e_b,
                'contra_fwd': c_f, 'contra_bwd': c_b,
                'composite_s': s, 'is_contradiction': contra,
            })
        inst_pairs[iid] = pairs

    for iid in instance_ids:
        pairs = inst_pairs[iid]
        measurable = len(pairs) >= 1 and inst_info[iid]['n_valid'] >= 2
        per_instance_rows.append({
            'model': MODEL_NAME, 'task': task, 'instance_id': iid,
            'n_valid_outputs': inst_info[iid]['n_valid'],
            'n_empty_outputs': inst_info[iid]['n_empty'],
            'n_pairs': len(pairs),
            'nli_composite_s':      float(np.mean([p['s'] for p in pairs])) if measurable else np.nan,
            'mean_entailment':      float(np.mean([p['e'] for p in pairs])) if measurable else np.nan,
            'mean_contradiction':   float(np.mean([p['c'] for p in pairs])) if measurable else np.nan,
            'frac_contra_pairs':    float(np.mean([p['contra'] for p in pairs])) if measurable else np.nan,
        })

# ------------------------------------------------------------------------
# Save
# ------------------------------------------------------------------------

pd.DataFrame(per_pair_rows).to_csv(
    os.path.join(OUT_DIR, f's3_nli_{MODEL_SLUG}_per_pair.csv'), index=False)

df = pd.DataFrame(per_instance_rows)
per_inst_csv = os.path.join(OUT_DIR, f's3_nli_{MODEL_SLUG}_per_instance.csv')
df.to_csv(per_inst_csv, index=False)

summary = (df.groupby('task')
             .agg(n_instances=('instance_id', 'count'),
                  n_low_valid=('n_valid_outputs', lambda s: int((s < 2).sum())),
                  mean_composite_s=('nli_composite_s', 'mean'),
                  median_composite_s=('nli_composite_s', 'median'),
                  mean_entailment=('mean_entailment', 'mean'),
                  mean_contradiction=('mean_contradiction', 'mean'),
                  frac_contra_pairs=('frac_contra_pairs', 'mean'),
                  p10_composite_s=('nli_composite_s', lambda s: s.quantile(0.10)),
                  p90_composite_s=('nli_composite_s', lambda s: s.quantile(0.90)))
             .round(4)
             .reset_index())
summary.insert(0, 'model', MODEL_NAME)
summary['nli_model'] = NLI_MODEL
summary['contra_threshold'] = CONTRA_THRESHOLD

summary_csv = os.path.join(OUT_DIR, f's3_nli_{MODEL_SLUG}_summary.csv')
summary.to_csv(summary_csv, index=False)

print(f"\n### S3 summary — {MODEL_NAME} ###")
print(summary.to_string(index=False))
print(f"\nSaved -> {per_inst_csv}")
print(f"      -> {summary_csv}")


Loading NLI model MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli on cuda ...
  labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}  (entail=0, contra=2)

=== aci_bench ===
  100 instances with all 5 seed outputs
  scoring 2000 directed pairs ...

=== meddialog ===
  500 instances with all 5 seed outputs
  scoring 10000 directed pairs ...

=== medicationqa ===
  500 instances with all 5 seed outputs
  scoring 10000 directed pairs ...

=== mtsamples ===
  500 instances with all 5 seed outputs
  scoring 9976 directed pairs ...

=== mtsamples_proc ===
  500 instances with all 5 seed outputs
  scoring 9940 directed pairs ...

### S3 summary — epfl-llm/meditron-7b ###
               model           task  n_instances  n_low_valid  mean_composite_s  median_composite_s  mean_entailment  mean_contradiction  frac_contra_pairs  p10_composite_s  p90_composite_s                                    nli_model  contra_threshold
epfl-llm/meditron-7b      aci_bench          100            0         

In [1]:
!pip install -U typing_extensions

In [2]:
!pip install "pydantic<2.10" && echo restart kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.2 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydantic]1/2 [pydantic]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistral-common 1.11.7 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.4.6 which is incompatible.
vllm 0.6.6 requires numpy<2.0.0, but you have numpy 2.4.6 which is incompatible.
restart kernel
